In [ ]:
!pip install tellurium

In [ ]:
pip install numpy==1.24.4

In [ ]:
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the full model
model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 1.0;     // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;     // Damaged epithelial cells
    species M1 in C1; M1 = 0.001;    // Recruited M1 macrophages
    species M2 in C1; M2 = 0.001;    // Recruited M2 macrophages
    species M0R in C1; M0R = 0.0;  // Resident M0 macrophages
    species M1R in C1; M1R = 0.0;   // Resident M1 macrophages
    species M2R in C1; M2R = 0.001;   // Resident M2 macrophages, low M2 dominant population
    species P in C1; P = 0.0;       // Pro-inflammatory cytokines
    species A in C1; A = 0.000;     // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.0001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m4 = 0.02;    // Macrophage decay (death/efflux) rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    sM0 = 0.1;    // Influx rate of M0 macrophages
    muM0 = 0.05;  // Efflux/death rate of M0 macrophages
    k_c = 0.0125; // Concentration of background anti-inflammatory cytokine

    K = 1;        // Carrying capacity
    WS = 1.0;       // Wood smoke exposure (1 = on, 0 = off)
    m11 = 0.2;    // Direct WS activation rate for M1
    m3 = 0.07;    // Resident macrophage proliferation rate
    smr = 0.1;    // Circulating macrophage source rate
    mumr = 0.05;  // Circulating macrophage decay rate

    // Helper functions for activation/influx
    RM1 := m1*(P^nm1/(Hm1^nm1 + P^nm1)) + m11*WS    // M1 activation function
    RM2 := m2*(A^nm2/(Hm2^nm2 + A^nm2)) + k_c       // M2 activation function

    // Differential equations
    Eh' = e1*(Eh + Ed)*(1 - (Eh + Ed)/K) - e2*Eh - e3*WS*Eh + e4*(1 + e9*(M2 + M2R) + e10*A)*Ed + e9*(M2 + M2R) + e10*A)*Ed
    Ed' = -e2*Ed + e3*WS*Eh - e4*Ed

    // Recruited macrophages
    M1' = WS*(smr*RM1)/(mumr + RM1 + RM2) - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M1
    M2' = WS*(smr*RM2)/(mumr + RM1 + RM2) + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M2

    // Resident macrophages
    M0R' = -m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m3*WS*(M1R + M2R)*(1 - (M0R + M1R + M2R)/K) - m4*M0R
    M1R' = m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M1R
    M2R' = m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M2R

    // Cytokine equations including both resident and recruited macrophages
    P' = e5*WS*(Eh^ne1/(He1^ne1 + Eh^ne1)) + e6*WS*(Ed^ne2/(He2^ne2 + Ed^ne2)) + m6*WS*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M1 + M1R) - m7*P
    A' = m8*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M2 + M2R) + m9*(Hm4^nm4/(A^nm4 + Hm4^nm4))*(M1 + M1R) - m10*A + k_c

  # // Wood smoke dosing event
  # E1: at (1 < time < 2): WS = 1.0;
  # E2: at (time >= 2): WS = 0.0;

  # // Wood smoke continuous exposure
  # E1: at (time > 1): WS = 1.0;



  // Wood smoke recurring dosing events for 3 meals a day
  E1: at (time % 24 >= 8 and time % 24 < 9): WS = 1.0;
  E2: at (time % 24 >= 9 and time % 24 < 13): WS = 0.0;
  E3: at (time % 24 >= 13 and time % 24 < 14): WS = 1.0;
  E4: at (time % 24 >= 14 and time % 24 < 18): WS = 0.0;
  E5: at (time % 24 >= 18 and time % 24 < 19): WS = 1.0;
  E6: at (time % 24 >= 19 and time % 24 < 24): WS = 0.0;
'''


# Load the model
r = te.loada(model)

# Define the scaling factors for pro-inflammatory cytokine production rates and damage rate
inflammation_scaling = [1.0, 0.75, 0.5, 0.25]  # 100%, 75%, 50%, 25% of original rates

# Define the scaling factors for repair rate
repair_scaling = [1.0, 1.25, 1.5, 1.75]  # 100%, 125%, 150%, 175% of original rate

# Set up the plot style
sns.set_style("whitegrid")
colors = sns.color_palette("colorblind")
linestyles = ['-', '--', '-.', ':']

# Species to plot
species = ['Eh', 'Ed', 'M1', 'M2', 'P', 'A']
labels = ['Healthy Epithelial', 'Damaged Epithelial',
          'M1 Macrophages', 'M2 Macrophages', 'Pro-inflammatory Cytokines',
          'Anti-inflammatory Cytokines']

# Perform parameter scan and plot
for j, (species_name, label) in enumerate(zip(species, labels)):
    plt.figure(figsize=(10, 6))

    for i in range(len(inflammation_scaling)):
        r.reset()

        infl_scale = inflammation_scaling[i]
        repair_scale = repair_scaling[i]

        # Scale the pro-inflammatory cytokine production rates and damage rate
        r.e5 = 0.1 * infl_scale
        r.e6 = 0.2 * infl_scale
        #r.m1 = 50 * infl_scale
        r.m6 = 0.2 * infl_scale
        r.e3 = r.e3 * infl_scale  # Decrease damage rate

        # Scale the repair rate
        r.e4 = r.e4 * repair_scale  # Increase repair rate
        r.m8 = r.m8 * repair_scale  # Increase anti
        r.m9 = r.m9 * repair_scale  # Increase anti

        # Run simulation
        result = r.simulate(0, 200, 1000)

        plt.plot(result[:, 0], result[:, j+1],
                 color=colors[i],
                 linestyle=linestyles[i],
                 label=f'{infl_scale*100}% inflammation, {repair_scale*100:.0f}% repair')

    plt.title(label)
    plt.xlabel('Time')
    plt.ylabel('Concentration')
    plt.legend(fontsize='small', loc='best')
    plt.tight_layout()
    plt.savefig(f'antioxidant_effect_{species_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the full model
model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 1.0;     // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;     // Damaged epithelial cells
    species M1 in C1; M1 = 0.001;    // Recruited M1 macrophages
    species M2 in C1; M2 = 0.001;    // Recruited M2 macrophages
    species M0R in C1; M0R = 0.0;  // Resident M0 macrophages
    species M1R in C1; M1R = 0.0;   // Resident M1 macrophages
    species M2R in C1; M2R = 0.001;   // Resident M2 macrophages, low M2 dominant population
    species P in C1; P = 0.0;       // Pro-inflammatory cytokines
    species A in C1; A = 0.000;     // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.0001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m4 = 0.02;    // Macrophage decay (death/efflux) rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    sM0 = 0.1;    // Influx rate of M0 macrophages
    muM0 = 0.05;  // Efflux/death rate of M0 macrophages
    k_c = 0.0125; // Concentration of background anti-inflammatory cytokine

    K = 1;        // Carrying capacity
    WS = 1.0;       // Wood smoke exposure (1 = on, 0 = off)
    m11 = 0.2;    // Direct WS activation rate for M1
    m3 = 0.07;    // Resident macrophage proliferation rate
    smr = 0.1;    // Circulating macrophage source rate
    mumr = 0.05;  // Circulating macrophage decay rate
    e8 = 0.0;   // Baseline A production by healthy epithelial cells
    e9 = 1.0;     // M2 enhancement of repair
    e10 = 1.0;    // A enhancement of repair


    // Helper functions for activation/influx
    RM1 := m1*(P^nm1/(Hm1^nm1 + P^nm1)) + m11*WS    // M1 activation function
    RM2 := m2*(A^nm2/(Hm2^nm2 + A^nm2)) + k_c       // M2 activation function

    // Differential equations
    Eh' = e1*(Eh + Ed)*(1 - (Eh + Ed)/K) - e2*Eh - e3*WS*Eh + e4*(1 + e9*(M2 + M2R) + e10*A)*Ed
    Ed' = -e2*Ed + e3*WS*Eh - e4*(1 + e9*(M2 + M2R) + e10*A)*Ed

    // Recruited macrophages
    M1' = WS*(smr*RM1)/(mumr + RM1 + RM2) - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M1
    M2' = WS*(smr*RM2)/(mumr + RM1 + RM2) + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M2

    // Resident macrophages
    M0R' = -m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m3*WS*(M1R + M2R)*(1 - (M0R + M1R + M2R)/K) - m4*M0R
    M1R' = m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M1R
    M2R' = m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M2R

    // Cytokine equations including both resident and recruited macrophages
    P' = e5*WS*(Eh^ne1/(He1^ne1 + Eh^ne1)) + e6*WS*(Ed^ne2/(He2^ne2 + Ed^ne2)) + m6*WS*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M1 + M1R) - m7*P
    A' = m8*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M2 + M2R) + m9*(Hm4^nm4/(A^nm4 + Hm4^nm4))*(M1 + M1R) - m10*A + k_c

  # // Wood smoke dosing event
  # E1: at (1 < time < 2): WS = 1.0;
  # E2: at (time >= 2): WS = 0.0;

  // Wood smoke continuous exposure
  E1: at (time > 1): WS = 1.0;



  # // Wood smoke recurring dosing events for 3 meals a day
  # E1: at (time % 24 >= 8 and time % 24 < 9): WS = 1.0;
  # E2: at (time % 24 >= 9 and time % 24 < 13): WS = 0.0;
  # E3: at (time % 24 >= 13 and time % 24 < 14): WS = 1.0;
  # E4: at (time % 24 >= 14 and time % 24 < 18): WS = 0.0;
  # E5: at (time % 24 >= 18 and time % 24 < 19): WS = 1.0;
  # E6: at (time % 24 >= 19 and time % 24 < 24): WS = 0.0;
'''


# Load the model
r = te.loada(model)

# Define the scaling factors for pro-inflammatory cytokine production rates and damage rate
inflammation_scaling = [1.0, 0.1, 0.01, 0.001]  #

# Define the scaling factors for repair rate
# repair_scaling = [1.0, 1.25, 1.5, 1.75]  #

# Set up the plot style
sns.set_style("whitegrid")
colors = sns.color_palette("colorblind")
linestyles = ['-', '--', '-.', ':']

# Plot cells
species = ['Eh', 'Ed', 'M1', 'M2', 'M0R', 'M1R', 'M2R', 'P', 'A']
labels = ['Healthy Epithelial', 'Damaged Epithelial', 'Recruited M1', 'Recruited M2',
               'Resident M0', 'Resident M1', 'Resident M2', 'Pro-inflammatory Cytokines',
          'Anti-inflammatory Cytokines']

# Perform parameter scan and plot
for j, (species_name, label) in enumerate(zip(species, labels)):
    plt.figure(figsize=(10, 6))

    for i in range(len(inflammation_scaling)):
        r.reset()

        infl_scale = inflammation_scaling[i]
        # repair_scale = repair_scaling[i]

        # Scale the pro-inflammatory cytokine production rates and damage rate
        r.e5 = 0.1 * infl_scale
        r.e6 = 0.2 * infl_scale
        #r.m1 = 50 * infl_scale
        r.m6 = 0.2 * infl_scale
        #r.e3 = r.e3 * infl_scale  # Decrease damage rate

        # Scale the repair rate
        # r.e4 = r.e4 * repair_scale  # Increase repair rate
        #r.m8 = r.m8 * repair_scale  # Increase anti
        #r.m9 = r.m9 * repair_scale  # Increase anti

        # Run simulation
        result = r.simulate(0, 200, 1000)

        # plt.plot(result[:, 0], result[:, j+1],
        #          color=colors[i],
        #          linestyle=linestyles[i],
        #          label=f'{infl_scale*100}% inflammation, {repair_scale*100:.0f}% repair')
        plt.plot(result[:, 0], result[:, j+1],
                 color=colors[i],
                 linestyle=linestyles[i],
                 label=f'{infl_scale} inflammation reduction factor')

    plt.title(label)
    plt.xlabel('Time')
    plt.ylabel('Concentration')
    plt.legend(fontsize='small', loc='best')
    plt.tight_layout()
    plt.savefig(f'antioxidant_effect_{species_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the full model
model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 1.0;     // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;     // Damaged epithelial cells
    species M1 in C1; M1 = 0.001;    // Recruited M1 macrophages
    species M2 in C1; M2 = 0.001;    // Recruited M2 macrophages
    species M0R in C1; M0R = 0.0;  // Resident M0 macrophages
    species M1R in C1; M1R = 0.0;   // Resident M1 macrophages
    species M2R in C1; M2R = 0.001;   // Resident M2 macrophages, low M2 dominant population
    species P in C1; P = 0.0;       // Pro-inflammatory cytokines
    species A in C1; A = 0.000;     // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.0001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m4 = 0.02;    // Macrophage decay (death/efflux) rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    sM0 = 0.1;    // Influx rate of M0 macrophages
    muM0 = 0.05;  // Efflux/death rate of M0 macrophages
    k_c = 0.0125; // Concentration of background anti-inflammatory cytokine

    K = 1;        // Carrying capacity
    WS = 1.0;       // Wood smoke exposure (1 = on, 0 = off)
    m11 = 0.2;    // Direct WS activation rate for M1
    m3 = 0.07;    // Resident macrophage proliferation rate
    smr = 0.1;    // Circulating macrophage source rate
    mumr = 0.05;  // Circulating macrophage decay rate
    e8 = 0.0;   // Baseline A production by healthy epithelial cells
    e9 = 1.0;     // M2 enhancement of repair
    e10 = 1.0;    // A enhancement of repair

    // Helper functions for activation/influx
    RM1 := m1*(P^nm1/(Hm1^nm1 + P^nm1)) + m11*WS    // M1 activation function
    RM2 := m2*(A^nm2/(Hm2^nm2 + A^nm2)) + k_c       // M2 activation function

    // Differential equations
    Eh' = e1*(Eh + Ed)*(1 - (Eh + Ed)/K) - e2*Eh - e3*WS*Eh + e4*(1 + e9*(M2 + M2R) + e10*A)*Ed
    Ed' = -e2*Ed + e3*WS*Eh - e4*(1 + e9*(M2 + M2R) + e10*A)*Ed

    // Recruited macrophages
    M1' = WS*(smr*RM1)/(mumr + RM1 + RM2) - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M1
    M2' = WS*(smr*RM2)/(mumr + RM1 + RM2) + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M2

    // Resident macrophages
    M0R' = -m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m3*WS*(M1R + M2R)*(1 - (M0R + M1R + M2R)/K) - m4*M0R
    M1R' = m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M1R
    M2R' = m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M2R

    // Cytokine equations including both resident and recruited macrophages
    P' = e5*WS*(Eh^ne1/(He1^ne1 + Eh^ne1)) + e6*WS*(Ed^ne2/(He2^ne2 + Ed^ne2)) + m6*WS*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M1 + M1R) - m7*P
    A' = m8*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M2 + M2R) + m9*(Hm4^nm4/(A^nm4 + Hm4^nm4))*(M1 + M1R) - m10*A + k_c

  // Wood smoke recurring dosing events for 3 meals a day
  E1: at (time % 24 >= 8 and time % 24 < 9): WS = 1.0;
  E2: at (time % 24 >= 9 and time % 24 < 13): WS = 0.0;
  E3: at (time % 24 >= 13 and time % 24 < 14): WS = 1.0;
  E4: at (time % 24 >= 14 and time % 24 < 18): WS = 0.0;
  E5: at (time % 24 >= 18 and time % 24 < 19): WS = 1.0;
  E6: at (time % 24 >= 19 and time % 24 < 24): WS = 0.0;
'''

# Load the model
r = te.loada(model)

# Set up parameter scans
inflammation_scaling = [1.0, 0.5, 0.25, 0.1]  # 0%, 50%, 75%, 90% reduction
anti_inflammatory_scaling = [1.0, 2.0, 4.0, 8.0]  # Up to 8x increase

# Set up the plot style
sns.set_style("whitegrid")
colors = sns.color_palette("colorblind")
linestyles = ['-', '--', '-.', ':']

# Define species and labels
species = ['Eh', 'Ed', 'M1', 'M2', 'M0R', 'M1R', 'M2R', 'P', 'A']
labels = ['Healthy Epithelial', 'Damaged Epithelial', 'Recruited M1', 'Recruited M2',
          'Resident M0', 'Resident M1', 'Resident M2', 'Pro-inflammatory Cytokines',
          'Anti-inflammatory Cytokines']

# Create subplots
fig, axs = plt.subplots(3, 3, figsize=(15, 15))
axs = axs.ravel()

# Perform parameter scan and plot
for j, (species_name, label) in enumerate(zip(species, labels)):
    ax = axs[j]

    for i in range(len(inflammation_scaling)):
        r.reset()

        infl_scale = inflammation_scaling[i]
        anti_scale = anti_inflammatory_scaling[i]

        # Scale pro-inflammatory production rates
        r.e5 = 0.1 * infl_scale  # Healthy epithelial pro-inflammatory production
        r.e6 = 0.2 * infl_scale  # Damaged epithelial pro-inflammatory production
        r.m6 = 0.2 * infl_scale  # M1 pro-inflammatory production

        # Scale anti-inflammatory parameters
        r.m8 = 0.1 * anti_scale  # M2 anti-inflammatory production
        r.m9 = 0.04 * anti_scale # M1 anti-inflammatory production
        r.k_c = 0.0125 * anti_scale  # Background anti-inflammatory level

        # Run simulation
        result = r.simulate(0, 200, 1000)

        ax.plot(result[:, 0], result[:, j+1],
                color=colors[i],
                linestyle=linestyles[i],
                label=f'{(1-infl_scale)*100:.0f}% ↓ pro-,\n{(anti_scale-1)*100:.0f}% ↑ anti-')

    ax.set_title(label)
    ax.set_xlabel('Time')
    ax.set_ylabel('Concentration')
    ax.legend(fontsize='x-small', loc='best')

plt.tight_layout()
plt.savefig('combined_treatment_effects.png', dpi=300, bbox_inches='tight')
plt.show()

# Also create individual plots for each species
for j, (species_name, label) in enumerate(zip(species, labels)):
    plt.figure(figsize=(10, 6))

    for i in range(len(inflammation_scaling)):
        r.reset()

        infl_scale = inflammation_scaling[i]
        anti_scale = anti_inflammatory_scaling[i]

        # Scale pro-inflammatory production rates
        r.e5 = 0.1 * infl_scale
        r.e6 = 0.2 * infl_scale
        r.m6 = 0.2 * infl_scale

        # Scale anti-inflammatory parameters
        r.m8 = 0.1 * anti_scale
        r.m9 = 0.04 * anti_scale
        r.k_c = 0.0125 * anti_scale

        # Run simulation
        result = r.simulate(0, 200, 1000)

        plt.plot(result[:, 0], result[:, j+1],
                 color=colors[i],
                 linestyle=linestyles[i],
                 label=f'{(1-infl_scale)*100:.0f}% reduction in pro-inflammatory,\n{(anti_scale-1)*100:.0f}% increase in anti-inflammatory')

    plt.title(label)
    plt.xlabel('Time')
    plt.ylabel('Concentration')
    plt.legend(fontsize='small', loc='best')
    plt.tight_layout()
    plt.savefig(f'treatment_effect_{species_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the full model (your original model code remains the same)
model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 1.0;     // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;     // Damaged epithelial cells
    species M1 in C1; M1 = 0.001;    // Recruited M1 macrophages
    species M2 in C1; M2 = 0.001;    // Recruited M2 macrophages
    species M0R in C1; M0R = 0.0;  // Resident M0 macrophages
    species M1R in C1; M1R = 0.0;   // Resident M1 macrophages
    species M2R in C1; M2R = 0.001;   // Resident M2 macrophages, low M2 dominant population
    species P in C1; P = 0.0;       // Pro-inflammatory cytokines
    species A in C1; A = 0.000;     // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.0001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m4 = 0.02;    // Macrophage decay (death/efflux) rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    sM0 = 0.1;    // Influx rate of M0 macrophages
    muM0 = 0.05;  // Efflux/death rate of M0 macrophages
    k_c = 0.0125; // Concentration of background anti-inflammatory cytokine

    K = 1;        // Carrying capacity
    WS = 1.0;     // Wood smoke exposure (1 = on, 0 = off)
    m11 = 0.2;    // Direct WS activation rate for M1
    m3 = 0.07;    // Resident macrophage proliferation rate
    smr = 0.1;    // Circulating macrophage source rate
    mumr = 0.05;  // Circulating macrophage decay rate
    e8 = 0.0;     // Baseline A production by healthy epithelial cells
    e9 = 1.0;     // M2 enhancement of repair
    e10 = 1.0;    // A enhancement of repair

    // Helper functions for activation/influx
    RM1 := m1*(P^nm1/(Hm1^nm1 + P^nm1)) + m11*WS    // M1 activation function
    RM2 := m2*(A^nm2/(Hm2^nm2 + A^nm2)) + k_c       // M2 activation function

    // Differential equations
    Eh' = e1*(Eh + Ed)*(1 - (Eh + Ed)/K) - e2*Eh - e3*WS*Eh + e4*(1 + e9*(M2 + M2R) + e10*A)*Ed
    Ed' = -e2*Ed + e3*WS*Eh - e4*(1 + e9*(M2 + M2R) + e10*A)*Ed

    // Recruited macrophages
    M1' = WS*(smr*RM1)/(mumr + RM1 + RM2) - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M1
    M2' = WS*(smr*RM2)/(mumr + RM1 + RM2) + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M2

    // Resident macrophages
    M0R' = -m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m3*WS*(M1R + M2R)*(1 - (M0R + M1R + M2R)/K) - m4*M0R
    M1R' = m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M1R
    M2R' = m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M2R

    // Cytokine equations including both resident and recruited macrophages
    P' = e5*WS*(Eh^ne1/(He1^ne1 + Eh^ne1)) + e6*WS*(Ed^ne2/(He2^ne2 + Ed^ne2)) + m6*WS*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M1 + M1R) - m7*P
    A' = m8*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M2 + M2R) + m9*(Hm4^nm4/(A^nm4 + Hm4^nm4))*(M1 + M1R) - m10*A + k_c

  # // Wood smoke recurring dosing events for 3 meals a day
  # E1: at (time % 24 >= 8 and time % 24 < 9): WS = 1.0;
  # E2: at (time % 24 >= 9 and time % 24 < 13): WS = 0.0;
  # E3: at (time % 24 >= 13 and time % 24 < 14): WS = 1.0;
  # E4: at (time % 24 >= 14 and time % 24 < 18): WS = 0.0;
  # E5: at (time % 24 >= 18 and time % 24 < 19): WS = 1.0;
  # E6: at (time % 24 >= 19 and time % 24 < 24): WS = 0.0;

  // Wood smoke dosing event
  E1: at (1 < time < 2): WS = 1.0;
  E2: at (time >= 2): WS = 0.0;
'''

# Load the model
r = te.loada(model)

# Set up parameter scans with fold changes
fold_changes = [1.0, 0.2, 0.1, 0.05]  # 1x, 1/5x, 1/10x, 1/20x for pro-inflammatory
anti_fold_changes = [1.0, 5.0, 10.0, 20.0]  # 1x, 5x, 10x, 20x for anti-inflammatory

# Set up the plot style
sns.set_style("whitegrid")
colors = sns.color_palette("colorblind")
linestyles = ['-', '--', '-.', ':']

# Define species and labels
species = ['Eh', 'Ed', 'M1', 'M2', 'M0R', 'M1R', 'M2R', 'P', 'A']
labels = ['Healthy Epithelial', 'Damaged Epithelial', 'Recruited M1', 'Recruited M2',
          'Resident M0', 'Resident M1', 'Resident M2', 'Pro-inflammatory Cytokines',
          'Anti-inflammatory Cytokines']

# Create subplots
fig, axs = plt.subplots(3, 3, figsize=(15, 15))
axs = axs.ravel()

# Perform parameter scan and plot
for j, (species_name, label) in enumerate(zip(species, labels)):
    ax = axs[j]

    for i in range(len(fold_changes)):
        r.reset()

        pro_scale = fold_changes[i]
        anti_scale = anti_fold_changes[i]

        # Scale pro-inflammatory parameters
        r.e5 = 0.1 * pro_scale   # Healthy epithelial pro-inflammatory production
        r.e6 = 0.2 * pro_scale   # Damaged epithelial pro-inflammatory production
        r.m6 = 0.2 * pro_scale   # M1 pro-inflammatory production
        r.m11 = 0.2 * pro_scale  # Direct WS activation of M1
        r.m1 = 50.0 * pro_scale  # M0 to M1 differentiation rate

        # Scale anti-inflammatory parameters
        r.m8 = 0.1 * anti_scale     # M2 anti-inflammatory production
        r.m9 = 0.04 * anti_scale    # M1 anti-inflammatory production
        r.k_c = 0.0125 * anti_scale # Background anti-inflammatory
        r.e9 = 1.0 * anti_scale     # M2 enhancement of repair
        r.e10 = 1.0 * anti_scale    # A enhancement of repair
        r.m2 = 0.1 * anti_scale     # M0 to M2 differentiation rate

        # Run simulation
        result = r.simulate(0, 200, 1000)

        ax.plot(result[:, 0], result[:, j+1],
                color=colors[i],
                linestyle=linestyles[i],
                label=f'{1/pro_scale:.0f}x ↓ pro-,\n{anti_scale:.0f}x ↑ anti-')

    ax.set_title(label)
    ax.set_xlabel('Time')
    ax.set_ylabel('Concentration')
    ax.legend(fontsize='x-small', loc='best')

plt.tight_layout()
plt.savefig('combined_treatment_effects.png', dpi=300, bbox_inches='tight')
plt.show()

# Also create individual plots for each species
for j, (species_name, label) in enumerate(zip(species, labels)):
    plt.figure(figsize=(10, 6))

    for i in range(len(fold_changes)):
        r.reset()

        pro_scale = fold_changes[i]
        anti_scale = anti_fold_changes[i]

        # Scale pro-inflammatory parameters
        r.e5 = 0.1 * pro_scale
        r.e6 = 0.2 * pro_scale
        r.m6 = 0.2 * pro_scale
        # r.m11 = 0.2 * pro_scale
        # r.m1 = 50.0 * pro_scale

        # Scale anti-inflammatory parameters
        r.m8 = 0.1 * anti_scale
        r.m9 = 0.04 * anti_scale
        r.k_c = 0.0125 * anti_scale
        r.e9 = 1.0 * anti_scale
        r.e10 = 1.0 * anti_scale
        # r.m2 = 0.1 * anti_scale

        # Run simulation
        result = r.simulate(0, 200, 1000)

        plt.plot(result[:, 0], result[:, j+1],
                 color=colors[i],
                 linestyle=linestyles[i],
                 label=f'{1/pro_scale:.0f}x reduction in pro-inflammatory,\n{anti_scale:.0f}x increase in anti-inflammatory')

    plt.title(label)
    plt.xlabel('Time')
    plt.ylabel('Concentration')
    plt.legend(fontsize='small', loc='best')
    plt.tight_layout()
    plt.savefig(f'treatment_effect_{species_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
def run_simulations_with_filtering(param_sets, max_threshold=1.0, min_samples=50):
    """
    Run simulations and filter out parameter sets that produce extreme values

    Parameters:
    - param_sets: list of parameter dictionaries
    - max_threshold: maximum allowed value for any variable
    - min_samples: minimum number of valid samples to collect

    Returns:
    - list of valid simulation results
    """
    print("Running simulations with filtering...")
    valid_results = []
    rejected_count = 0
    total_tried = 0

    # Process parameter sets until we have enough valid samples
    for params in param_sets:
        total_tried += 1

        # Run simulation
        result = run_simulation(params)

        # Check if any variable exceeds threshold
        exceeds_threshold = False
        for col_idx in range(1, result.shape[1]):  # Skip time column
            if np.max(result[:, col_idx]) > max_threshold:
                exceeds_threshold = True
                break

        if exceeds_threshold:
            rejected_count += 1
        else:
            valid_results.append(result)

        # Report progress
        if total_tried % 10 == 0:
            print(f"Processed {total_tried} parameter sets, accepted {len(valid_results)}, rejected {rejected_count}")

        # Check if we have enough valid samples
        if len(valid_results) >= min_samples:
            break

    # Final report
    print(f"Simulation complete. Total tried: {total_tried}, accepted: {len(valid_results)}, rejected: {rejected_count}")

    return valid_results

# Generate more parameter sets than needed to account for rejection
n_samples_to_generate = 200  # Generate 2x what we want to allow for rejections
sampler = qmc.LatinHypercube(d=len(param_ranges))
samples = sampler.random(n=n_samples_to_generate)

# Scale samples to parameter ranges
param_names = list(param_ranges.keys())
all_param_sets = []
for i in range(n_samples_to_generate):
    param_set = {}
    for j, param in enumerate(param_names):
        min_val, max_val = param_ranges[param]
        param_set[param] = min_val + samples[i,j] * (max_val - min_val)
    all_param_sets.append(param_set)

# Run simulations with filtering
max_threshold = 10.0
min_samples = 50  # Minimum number of valid samples we want
valid_results = run_simulations_with_filtering(all_param_sets, max_threshold, min_samples)

def select_representative_trajectories(all_results, variables, n_representatives=3):
    """
    Select representative trajectories that show different qualitative behaviors

    Parameters:
    - all_results: list of simulation results
    - variables: list of variable names
    - n_representatives: number of representative trajectories to select

    Returns:
    - indices of representative trajectories
    """
    # Initialize dictionary to store features for clustering
    features = {}
    n_samples = len(all_results)

    # Extract key features for each trajectory
    for var_idx, var_name in enumerate(variables):
        # Get all trajectories for this variable
        trajectories = np.array([result[:,var_idx+1] for result in all_results])

        # Calculate features for each trajectory
        max_values = np.max(trajectories, axis=1)
        peak_times = np.argmax(trajectories, axis=1)
        final_values = trajectories[:, -1]

        # Store features
        features[f"{var_name}_max"] = max_values
        features[f"{var_name}_peak_time"] = peak_times
        features[f"{var_name}_final"] = final_values

    # Convert features to matrix
    feature_matrix = np.column_stack([features[key] for key in features.keys()])

    # Normalize features to [0, 1] range
    feature_mins = np.min(feature_matrix, axis=0)
    feature_maxs = np.max(feature_matrix, axis=0)
    feature_range = feature_maxs - feature_mins
    # Avoid division by zero
    feature_range[feature_range == 0] = 1.0
    normalized_features = (feature_matrix - feature_mins) / feature_range

    # Use a simple distance-based approach to select diverse trajectories
    selected_indices = []
    remaining_indices = list(range(n_samples))

    # Start with the median trajectory
    median_idx = remaining_indices[len(remaining_indices) // 2]
    selected_indices.append(median_idx)
    remaining_indices.remove(median_idx)

    # Find other trajectories that are most different from already selected ones
    while len(selected_indices) < n_representatives and remaining_indices:
        max_min_distance = -1
        farthest_idx = -1

        for idx in remaining_indices:
            # Calculate minimum distance to any selected trajectory
            min_distance = float('inf')
            for selected_idx in selected_indices:
                dist = np.linalg.norm(normalized_features[idx] - normalized_features[selected_idx])
                min_distance = min(min_distance, dist)

            # Update if this is the farthest candidate found
            if min_distance > max_min_distance:
                max_min_distance = min_distance
                farthest_idx = idx

        selected_indices.append(farthest_idx)
        remaining_indices.remove(farthest_idx)

    return selected_indices

# Plot with highlighted representatives
def plot_with_highlighted_representatives(all_results, variables, n_representatives=3):
    """
    Plot with highlighted representative trajectories using dashed lines
    """
    # Select representative trajectories
    rep_indices = select_representative_trajectories(all_results, variables, n_representatives)

    # Define colors for representative trajectories
    rep_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, orange, green
    background_color = '#DDDDDD'  # Light gray
    percentile_color = (0.7, 0.7, 0.7)  # Medium gray

    for var_idx, var_name in enumerate(variables):
        plt.figure(figsize=(10, 6))

        # Get time and all trajectories
        time = all_results[0][:,0]
        trajectories = np.array([result[:,var_idx+1] for result in all_results])

        # Plot background trajectories in light gray
        for i in range(len(all_results)):
            if i not in rep_indices:
                plt.plot(time, trajectories[i], color=background_color, alpha=1.0, linewidth=0.3)

        # Calculate and plot percentiles
        median = np.median(trajectories, axis=0)
        p25 = np.percentile(trajectories, 25, axis=0)
        p75 = np.percentile(trajectories, 75, axis=0)

        plt.fill_between(time, p25, p75, color=percentile_color, alpha=0.3,
                        label='25th-75th percentile')

        # Plot representative trajectories with dashed lines
        for i, idx in enumerate(rep_indices):
            plt.plot(time, trajectories[idx], color=rep_colors[i],
                    linestyle='--', linewidth=2.0, label='Representative trajectory')

        # Add median curve (solid line)
        plt.plot(time, median, 'k-', linewidth=1.5, label='Median trajectory')

        # Remove duplicate labels in legend
        handles, labels = plt.gca().get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        plt.legend(by_label.values(), by_label.keys(), fontsize=10)

        plt.xlabel('Time (hours)', fontsize=12)
        plt.ylabel(f'{var_name} Concentration', fontsize=12)
        plt.grid(False)

        # Set axis limits
        plt.xlim(0, 200)
        # Calculate max value of representative trajectories
        rep_max = max([np.max(trajectories[idx]) for idx in rep_indices])
        # Calculate 95th percentile of all trajectories
        p95_all = np.percentile(trajectories, 95)
        # Set y-axis limit to ensure all representative trajectories are visible
        # with a small buffer (10%) and a minimum value of 0.1
        plt.ylim(0, max(rep_max * 1.1, p95_all, 0.1))

        # Save plots
        plt.savefig(f'LHS_{var_name}_representatives.png', dpi=300, bbox_inches='tight')
        #plt.savefig(f'LHS_{var_name}_representatives.eps', dpi=300, bbox_inches='tight')
        plt.close()

# Plot results
variables = ['Eh', 'Ed', 'M1', 'M2', 'M0R', 'M1R', 'M2R', 'P', 'A']
plot_with_highlighted_representatives(valid_results, variables, n_representatives=3)

In [ ]:
def select_representative_trajectories(all_results, variables, n_representatives=3):
    """
    Select representative trajectories that show different qualitative behaviors

    Parameters:
    - all_results: list of simulation results
    - variables: list of variable names
    - n_representatives: number of representative trajectories to select

    Returns:
    - indices of representative trajectories
    """
    # Initialize dictionary to store features for clustering
    features = {}
    n_samples = len(all_results)

    # Extract key features for each trajectory
    for var_idx, var_name in enumerate(variables):
        # Get all trajectories for this variable
        trajectories = np.array([result[:,var_idx+1] for result in all_results])

        # Calculate features for each trajectory
        max_values = np.max(trajectories, axis=1)
        peak_times = np.argmax(trajectories, axis=1)
        final_values = trajectories[:, -1]

        # Store features
        features[f"{var_name}_max"] = max_values
        features[f"{var_name}_peak_time"] = peak_times
        features[f"{var_name}_final"] = final_values

    # Convert features to matrix
    feature_matrix = np.column_stack([features[key] for key in features.keys()])

    # Normalize features to [0, 1] range
    feature_mins = np.min(feature_matrix, axis=0)
    feature_maxs = np.max(feature_matrix, axis=0)
    feature_range = feature_maxs - feature_mins
    # Avoid division by zero
    feature_range[feature_range == 0] = 1.0
    normalized_features = (feature_matrix - feature_mins) / feature_range

    # Use a simple distance-based approach to select diverse trajectories
    selected_indices = []
    remaining_indices = list(range(n_samples))

    # Start with the median trajectory
    median_idx = remaining_indices[len(remaining_indices) // 2]
    selected_indices.append(median_idx)
    remaining_indices.remove(median_idx)

    # Find other trajectories that are most different from already selected ones
    while len(selected_indices) < n_representatives and remaining_indices:
        max_min_distance = -1
        farthest_idx = -1

        for idx in remaining_indices:
            # Calculate minimum distance to any selected trajectory
            min_distance = float('inf')
            for selected_idx in selected_indices:
                dist = np.linalg.norm(normalized_features[idx] - normalized_features[selected_idx])
                min_distance = min(min_distance, dist)

            # Update if this is the farthest candidate found
            if min_distance > max_min_distance:
                max_min_distance = min_distance
                farthest_idx = idx

        selected_indices.append(farthest_idx)
        remaining_indices.remove(farthest_idx)

    return selected_indices

def plot_with_highlighted_representatives(all_results, variables, n_representatives=3):
    """
    Plot with highlighted representative trajectories
    """
    # Select representative trajectories
    rep_indices = select_representative_trajectories(all_results, variables, n_representatives)

    # Define colors for representative trajectories
    rep_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, orange, green
    background_color = '#DDDDDD'  # Light gray
    percentile_color = (0.7, 0.7, 0.7)  # Medium gray

    for var_idx, var_name in enumerate(variables):
        plt.figure(figsize=(10, 6))

        # Get time and all trajectories
        time = all_results[0][:,0]
        trajectories = np.array([result[:,var_idx+1] for result in all_results])

        # Plot background trajectories in light gray
        for i in range(len(all_results)):
            if i not in rep_indices:
                plt.plot(time, trajectories[i], color=background_color, alpha=0.3, linewidth=0.5)

        # Calculate and plot percentiles
        median = np.median(trajectories, axis=0)
        p25 = np.percentile(trajectories, 25, axis=0)
        p75 = np.percentile(trajectories, 75, axis=0)

        plt.fill_between(time, p25, p75, color=percentile_color, alpha=0.3,
                        label='25th-75th percentile')

        # Plot representative trajectories with distinct colors and thicker lines
        for i, idx in enumerate(rep_indices):
            plt.plot(time, trajectories[idx], color=rep_colors[i],
                    linewidth=2.5, label=f'Representative {i+1}')

        # Add median curve
        plt.plot(time, median, 'k-', linewidth=1.5, label='Median')

        plt.xlabel('Time (hours)', fontsize=12)
        plt.ylabel(f'{var_name} Concentration', fontsize=12)
        plt.legend(fontsize=10)
        plt.grid(False)

        # Set axis limits
        plt.xlim(0, 200)
        plt.ylim(bottom=0)

        # Save plots
        plt.savefig(f'LHS_{var_name}_representatives.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'LHS_{var_name}_representatives.eps', dpi=300, bbox_inches='tight')
        plt.close()

# Example usage
n_representatives = 3  # Number of highlighted trajectories
plot_with_highlighted_representatives(all_results, variables, n_representatives)

In [ ]:
def select_representative_trajectories(all_results, variables, n_representatives=3, max_threshold=2.0):
    """
    Select representative trajectories that show different qualitative behaviors,
    excluding trajectories with values exceeding max_threshold

    Parameters:
    - all_results: list of simulation results
    - variables: list of variable names
    - n_representatives: number of representative trajectories to select
    - max_threshold: maximum allowed value for representative trajectories

    Returns:
    - indices of representative trajectories
    """
    # Initialize dictionary to store features for clustering
    features = {}
    n_samples = len(all_results)

    # Identify trajectories exceeding threshold in any variable
    excluded_indices = set()
    for var_idx, var_name in enumerate(variables):
        trajectories = np.array([result[:,var_idx+1] for result in all_results])
        max_values = np.max(trajectories, axis=1)
        # Identify indices where max value exceeds threshold
        for i, max_val in enumerate(max_values):
            if max_val > max_threshold:
                excluded_indices.add(i)

    # Extract key features for each trajectory
    for var_idx, var_name in enumerate(variables):
        # Get all trajectories for this variable
        trajectories = np.array([result[:,var_idx+1] for result in all_results])

        # Calculate features for each trajectory
        max_values = np.max(trajectories, axis=1)
        peak_times = np.argmax(trajectories, axis=1)
        final_values = trajectories[:, -1]

        # Store features
        features[f"{var_name}_max"] = max_values
        features[f"{var_name}_peak_time"] = peak_times
        features[f"{var_name}_final"] = final_values

    # Convert features to matrix
    feature_matrix = np.column_stack([features[key] for key in features.keys()])

    # Normalize features to [0, 1] range
    feature_mins = np.min(feature_matrix, axis=0)
    feature_maxs = np.max(feature_matrix, axis=0)
    feature_range = feature_maxs - feature_mins
    # Avoid division by zero
    feature_range[feature_range == 0] = 1.0
    normalized_features = (feature_matrix - feature_mins) / feature_range

    # Use a simple distance-based approach to select diverse trajectories
    selected_indices = []
    remaining_indices = [i for i in range(n_samples) if i not in excluded_indices]

    if not remaining_indices:
        print("Warning: All trajectories exceed max_threshold. Using all trajectories.")
        remaining_indices = list(range(n_samples))

    # Start with the median trajectory (from non-excluded ones)
    median_idx = remaining_indices[len(remaining_indices) // 2]
    selected_indices.append(median_idx)
    remaining_indices.remove(median_idx)

    # Find other trajectories that are most different from already selected ones
    while len(selected_indices) < n_representatives and remaining_indices:
        max_min_distance = -1
        farthest_idx = -1

        for idx in remaining_indices:
            # Calculate minimum distance to any selected trajectory
            min_distance = float('inf')
            for selected_idx in selected_indices:
                dist = np.linalg.norm(normalized_features[idx] - normalized_features[selected_idx])
                min_distance = min(min_distance, dist)

            # Update if this is the farthest candidate found
            if min_distance > max_min_distance:
                max_min_distance = min_distance
                farthest_idx = idx

        selected_indices.append(farthest_idx)
        remaining_indices.remove(farthest_idx)

    return selected_indices

def plot_with_highlighted_representatives(all_results, variables, n_representatives=3, max_threshold=1.0):
    """
    Plot with highlighted representative trajectories, using dashed lines for representatives
    and excluding trajectories with values exceeding max_threshold
    """
    # Select representative trajectories
    rep_indices = select_representative_trajectories(all_results, variables, n_representatives, max_threshold)

    # Define colors for representative trajectories
    rep_colors = ['#1f77b4', '#ff7f0e', '#2ca02c']  # Blue, orange, green
    background_color = '#DDDDDD'  # Light gray
    percentile_color = (0.7, 0.7, 0.7)  # Medium gray

    for var_idx, var_name in enumerate(variables):
        plt.figure(figsize=(10, 6))

        # Get time and all trajectories
        time = all_results[0][:,0]
        trajectories = np.array([result[:,var_idx+1] for result in all_results])

        # Determine appropriate y-axis limit for this variable
        # First calculate 95th percentile of all values across all timepoints and trajectories
        p95_value = np.percentile(trajectories, 95)
        # Set y_max to a reasonable value that contains most data while hiding extreme outliers
        y_max = min(p95_value * 1.2, max_threshold)

        # Plot background trajectories in light gray
        for i in range(len(all_results)):
            if i not in rep_indices:
                plt.plot(time, trajectories[i], color=background_color, alpha=0.9, linewidth=0.3)

        # Calculate and plot percentiles
        median = np.median(trajectories, axis=0)
        p25 = np.percentile(trajectories, 25, axis=0)
        p75 = np.percentile(trajectories, 75, axis=0)

        plt.fill_between(time, p25, p75, color=percentile_color, alpha=0.3,
                        label='25th-75th percentile')

        # Plot representative trajectories with dashed lines
        for i, idx in enumerate(rep_indices):
            plt.plot(time, trajectories[idx], color=rep_colors[i],
                    linestyle='--', linewidth=2.0, label=f'Representative trajectory')

        # Add median curve (solid line)
        plt.plot(time, median, 'k-', linewidth=1.5, label='Median trajectory')

        # Remove duplicate labels in legend
        handles, labels = plt.gca().get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        plt.legend(by_label.values(), by_label.keys(), fontsize=10)

        plt.xlabel('Time (hours)', fontsize=12)
        plt.ylabel(f'{var_name} Concentration', fontsize=12)
        plt.grid(False)

        # Set axis limits
        plt.xlim(0, 200)
        #plt.ylim(0, y_max)

        # Save plots
        plt.savefig(f'LHS_{var_name}_representatives.png', dpi=300, bbox_inches='tight')
        plt.savefig(f'LHS_{var_name}_representatives.eps', dpi=300, bbox_inches='tight')
        plt.close()

# Example usage
n_representatives = 3  # Number of highlighted trajectories
max_threshold = 10.0   # Maximum value allowed for representative trajectories
plot_with_highlighted_representatives(all_results, variables, n_representatives, max_threshold)

In [ ]:
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt

def calculate_prcc_multitimepoint(parameter_values, output_values, param_names, time_points):
    """
    Calculate PRCCs at multiple timepoints
    """
    n_params = parameter_values.shape[1]
    n_times = len(time_points)
    prcc_values = np.zeros((n_params, n_times))
    p_values = np.zeros((n_params, n_times))

    for t in range(n_times):
        # Convert inputs to ranks
        param_ranks = np.array([stats.rankdata(parameter_values[:,j])
                              for j in range(n_params)]).T
        output_ranks = stats.rankdata(output_values[:,t])

        # Calculate residuals for each parameter
        param_residuals = np.zeros_like(param_ranks)
        for i in range(n_params):
            mask = np.ones(n_params, bool)
            mask[i] = False
            param_residuals[:,i] = calculate_residuals(param_ranks[:,i], param_ranks[:,mask])

        # Calculate residuals for output
        output_residuals = np.zeros(len(output_ranks))
        for i in range(n_params):
            mask = np.ones(n_params, bool)
            mask[i] = False
            output_residuals = calculate_residuals(output_ranks, param_ranks[:,mask])

        # Calculate PRCC and p-values
        for i in range(n_params):
            correlation, pvalue = stats.pearsonr(param_residuals[:,i], output_residuals)
            prcc_values[i,t] = correlation
            p_values[i,t] = pvalue

    return prcc_values, p_values

def calculate_residuals(y, X):
    """Calculate residuals from linear regression"""
    X = np.column_stack([np.ones(len(X)), X])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    y_pred = X @ beta
    return y - y_pred

# Convert parameter sets to array format
param_names = list(param_ranges.keys())
param_array = np.array([[params[param] for param in param_names]
                       for params in param_sets])

# Define variables to analyze
variables = {
    'Ed': 2,  # Damaged epithelial cells
    'P': 8,   # Pro-inflammatory cytokines
    'A': 9,   # Anti-inflammatory cytokines
    'M1': 3,  # M1 macrophages
    'M2': 4   # M2 macrophages
}

# Select specific timepoints for analysis (e.g., early, middle, and late responses)
time_indices = [10, 50, 100, -1]  # early, middle, late, and final timepoints
time_points = all_results[0][time_indices,0]

# Calculate PRCCs for each variable at multiple timepoints
for var_name, var_idx in variables.items():
    print(f"\nAnalyzing {var_name}:")

    # Extract variable data for all timepoints
    var_outputs = np.array([result[:,var_idx] for result in all_results])
    var_outputs_selected = var_outputs[:,time_indices]

    # Calculate PRCCs
    prcc_values, p_values = calculate_prcc_multitimepoint(param_array, var_outputs_selected,
                                                         param_names, time_points)

    # Create heatmap for this variable
    plt.figure(figsize=(10, 12))
    sns.heatmap(prcc_values,
                xticklabels=[f't={t:.1f}' for t in time_points],
                yticklabels=param_names,
                cmap='RdBu',
                center=0,
                vmin=-1,
                vmax=1)
    plt.title(f'PRCC Values for {var_name} across time')
    plt.tight_layout()
    plt.savefig(f'prcc_{var_name}_time.png', dpi=300, bbox_inches='tight')
    plt.close()

    # Print top 5 most influential parameters at each timepoint
    for t, time in enumerate(time_points):
        print(f"\nTime = {time:.1f}:")
        # Sort parameters by absolute PRCC value
        sorted_indices = np.argsort(np.abs(prcc_values[:,t]))[::-1]

        for i in range(5):
            idx = sorted_indices[i]
            print(f"{param_names[idx]}: PRCC = {prcc_values[idx,t]:.3f} "
                  f"(p = {p_values[idx,t]:.3e})")

# Calculate and print maximum absolute PRCC for each parameter-variable pair
print("\nMaximum absolute PRCC values across all timepoints:")
for var_name, var_idx in variables.items():
    print(f"\n{var_name}:")
    var_outputs = np.array([result[:,var_idx] for result in all_results])
    var_outputs_selected = var_outputs[:,time_indices]

    prcc_values, p_values = calculate_prcc_multitimepoint(param_array, var_outputs_selected,
                                                         param_names, time_points)

    # Get maximum absolute PRCC and corresponding timepoint for each parameter
    max_abs_prcc = np.max(np.abs(prcc_values), axis=1)
    max_abs_indices = np.argmax(np.abs(prcc_values), axis=1)

    # Sort parameters by maximum absolute PRCC
    sorted_indices = np.argsort(max_abs_prcc)[::-1]

    # Print top 10 parameters
    for idx in sorted_indices[:10]:
        t = max_abs_indices[idx]
        print(f"{param_names[idx]}: max|PRCC| = {max_abs_prcc[idx]:.3f} "
              f"at t = {time_points[t]:.1f} (p = {p_values[idx,t]:.3e})")

In [ ]:
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 1.0;     // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;     // Damaged epithelial cells
    species M1 in C1; M1 = 0.001;    // Recruited M1 macrophages
    species M2 in C1; M2 = 0.001;    // Recruited M2 macrophages
    species M0R in C1; M0R = 0.0;  // Resident M0 macrophages
    species M1R in C1; M1R = 0.0;   // Resident M1 macrophages
    species M2R in C1; M2R = 0.001;   // Resident M2 macrophages
    species P in C1; P = 0.0;       // Pro-inflammatory cytokines
    species A in C1; A = 0.000;     // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.0001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m4 = 0.02;    // Macrophage death rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    sM0 = 0.1;    // Influx rate of M0 macrophages
    muM0 = 0.05;  // Efflux/death rate of M0 macrophages
    k_c = 0.0125; // Background anti-inflammatory cytokine

    K = 1;        // Carrying capacity
    WS = 1.0;     // Wood smoke exposure (1 = on, 0 = off)
    m11 = 0.2;    // Direct WS activation rate for M1
    m3 = 0.07;    // Resident macrophage proliferation rate
    smr = 0.1;    // Circulating macrophage source rate
    mumr = 0.05;  // Circulating macrophage decay rate
    e8 = 0.0;     // Baseline A production by healthy epithelial cells
    e9 = 1.0;     // M2 enhancement of repair
    e10 = 1.0;    // A enhancement of repair

    // Helper functions
    RM1 := m1*(P^nm1/(Hm1^nm1 + P^nm1)) + m11*WS    // M1 activation function
    RM2 := m2*(A^nm2/(Hm2^nm2 + A^nm2)) + k_c       // M2 activation function

    // Differential equations with corrected repair terms
    Eh' = e1*(Eh + Ed)*(1 - (Eh + Ed)/K) - e2*Eh - e3*WS*Eh + e4*(1 + e9*(M2 + M2R) + e10*A)*Ed
    Ed' = -e2*Ed + e3*WS*Eh - e4*(1 + e9*(M2 + M2R) + e10*A)*Ed

    M1' = WS*(smr*RM1)/(mumr + RM1 + RM2) - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M1
    M2' = WS*(smr*RM2)/(mumr + RM1 + RM2) + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M2

    M0R' = -m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m3*WS*(M1R + M2R)*(1 - (M0R + M1R + M2R)/K) - m4*M0R
    M1R' = m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M1R
    M2R' = m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M2R

    P' = e5*WS*(Eh^ne1/(He1^ne1 + Eh^ne1)) + e6*WS*(Ed^ne2/(He2^ne2 + Ed^ne2)) + m6*WS*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M1 + M1R) - m7*P
    A' = m8*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M2 + M2R) + m9*(Hm4^nm4/(A^nm4 + Hm4^nm4))*(M1 + M1R) - m10*A + k_c

    // Wood smoke dosing event
    E1: at (1 < time < 2): WS = 1.0;
    E2: at (time >= 2): WS = 0.0;
'''

# Load model
r = te.loada(model)

# Set up IL-10 deficiency parameter scan
il10_reductions = [1.0, 0.1]  # Normal and 90% reduction

# Set up plot style
sns.set_style("whitegrid")
colors = sns.color_palette("colorblind")
linestyles = ['-', '--']

# Define species and labels
species = ['Eh', 'Ed', 'M1', 'M2', 'M0R', 'M1R', 'M2R', 'P', 'A']
labels = ['Healthy Epithelial', 'Damaged Epithelial', 'Recruited M1', 'Recruited M2',
          'Resident M0', 'Resident M1', 'Resident M2', 'Pro-inflammatory Cytokines',
          'Anti-inflammatory Cytokines']

# Create individual plots for each species
for j, (species_name, label) in enumerate(zip(species, labels)):
    plt.figure(figsize=(10, 6))

    for i in range(len(il10_reductions)):
        r.reset()

        scale = il10_reductions[i]

        # Scale down IL-10 production parameters
        r.m8 = 0.1 * scale     # M2 anti-inflammatory production
        r.m9 = 0.04 * scale    # M1 anti-inflammatory production
        r.k_c = 0.0125 * scale # Background anti-inflammatory

        # Run simulation
        result = r.simulate(0, 200, 1000)

        plt.plot(result[:, 0], result[:, j+1],
                color=colors[i],
                linestyle=linestyles[i],
                label=f'{scale:.2f}x IL-10 production')

    plt.title(label)
    plt.xlabel('Time')
    plt.ylabel(f'{label} Concentration')
    plt.legend(fontsize='small')
    plt.tight_layout()
    plt.savefig(f'il10_deficiency_{species_name}.eps', dpi=300, bbox_inches='tight')
    plt.show()